# Text Summarization with TextRank and Neural Methods

This notebook provides an educational overview of automatic text summarization techniques, focusing on the TextRank algorithm and its application to the CNN/DailyMail dataset. We'll cover preprocessing steps, evaluation with ROUGE, and compare TextRank with frequency-based methods.

## 1. Introduction to Text Summarization

Text summarization is the process of condensing a source text into a shorter version while preserving its essential meaning and information content. There are two main approaches:

- **Extractive summarization**: Selects important sentences or passages directly from the source text and combines them to form a summary.
- **Abstractive summarization**: Generates new text that may use different words and phrasing to convey the core ideas of the source text.

This notebook focuses on **extractive summarization** using the TextRank algorithm, which is based on graph-based ranking algorithms.

## 2. The TextRank Algorithm

TextRank is a graph-based ranking algorithm inspired by Google's PageRank. It treats sentences as nodes in a graph, where edges represent semantic similarity between sentences.

### How TextRank Works:

1. **Sentence Extraction**: Each sentence is represented as a vector (e.g., using TF-IDF or word embeddings).
2. **Similarity Matrix**: Compute pairwise similarity between all sentences using cosine similarity.
3. **Graph Construction**: Sentences are nodes, and edges exist between sentences with similarity above a threshold.
4. **Ranking**: Apply PageRank algorithm to compute the importance score for each sentence.
5. **Summary Selection**: Select top-k sentences with highest scores as the summary.

The PageRank algorithm can be defined by the following recursive formula:

$$PR(i) = \frac{1-d}{N} + d \sum_{j \in S_i} \frac{PR(j)}{L(j)}$$

Where:
- $PR(i)$ is the PageRank of page $i$
- $d$ is the damping factor (usually 0.85)
- $N$ is the total number of pages
- $S_i$ is the set of pages that link to page $i$
- $L(j)$ is the number of outgoing links from page $j$

In [ ]:
# TextRank Implementation

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx
from collections import Counter

def textrank_summarize(text, num_summaries=3, damping=0.85, max_iter=100, tol=1e-6):
    """Extractive summarization using TextRank algorithm.
    
    Args:
        text: Input text
        num_summaries: Number of sentences to include in summary
        damping: Damping factor for PageRank
        max_iter: Maximum number of iterations for PageRank
        tol: Tolerance for convergence
    """
    
    # Tokenize sentences
    sentences = [s.strip() for s in text.split('. ') if s]
    
    # Create TF-IDF matrix
    vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
    tfidf_matrix = vectorizer.fit_transform(sentences)
    
    # Compute sentence similarity matrix
    similarity_matrix = cosine_similarity(tfidf_matrix)
    
    # Create graph from similarity matrix
    nx_graph = nx.from_numpy_array(similarity_matrix)
    
    # Calculate PageRank scores
    scores = nx.pagerank(nx_graph, alpha=damping, max_iter=max_iter, tol=tol)
    
    # Rank sentences by score
    ranked_sentences = sorted(((scores[i], i, s) for i, s in enumerate(sentences)), reverse=True)
    
    # Select top-k sentences
    top_sentences = [s for _, i, s in ranked_sentences[:num_summaries]]
    
    # Sort selected sentences by their original order
    top_indices = [ranked_sentences[i][1] for i in range(num_summaries)]
    top_indices.sort()
    summary = '. '.join([sentences[i] for i in top_indices]) + '.'
    
    return summary, ranked_sentences[:num_summaries]

# Example usage
sample_text = """.
Natural language processing (NLP) is a subfield of linguistics, computer science, and artificial intelligence concerned with the interactions between computers and human language.
It involves programming computers to process and analyze large amounts of natural language data.
The goal is to enable computers to understand, interpret, and generate human language in a valuable way.
NLP is used in many applications such as machine translation, sentiment analysis, and text summarization.
Text summarization, specifically, aims to condense the content of a text while preserving its main ideas.
There are two primary types: extractive and abstractive summarization.
Extractive methods select important sentences directly from the source, while abstractive methods generate new text.
TextRank is a popular extractive method based on graph-based ranking algorithms.
It treats sentences as nodes in a graph and uses similarity to create edges between them.
The PageRank algorithm then determines the most important sentences.
"""

summary, rankings = textrank_summarize(sample_text, num_summaries=2)
print("Sample Summary:")
print(summary)
print("\nTop Ranked Sentences:")
for score, idx, sentence in rankings[:3]:
    print(f"Score: {score:.4f} - {sentence[:60]}...")


## 3. CNN/DailyMail Dataset

The CNN/Daily Mail dataset is one of the most widely used benchmarks for text summarization research. It contains:

- **CNN**: 93,000 news articles (93k training, 2.5k validation, 2.5k test)
- **Daily Mail**: 220,000 news articles (200k training, 12.5k validation, 12.5k test)
- **Total**: Approximately 313,000 news articles

### Dataset Characteristics:

- Each article is paired with a human-written summary (highlights) that appears at the top of the article
- Articles are typically 500-1000 words, summaries are 3-5 sentences (about 3-4 lines)
- The dataset is preprocessed to remove low-quality examples
- It's available in both raw and tokenized formats

### Why This Dataset?

1. **Large scale**: Enables training of data-hungry neural models
2. **High quality**: Professionally written summaries by journalists
3. **Domain consistency**: All articles are news stories, making it easier to learn domain-specific patterns
4. **Standard benchmark**: Allows fair comparison between different summarization methods

### Loading the Dataset

In practice, you would load the dataset using Hugging Face datasets library:

In [ ]:
# Loading CNN/DailyMail dataset

from datasets import load_dataset

# Load the dataset
dataset = load_dataset('cnn_dailymail', '3.0.0', split='train[:100]')  # Using a small sample for demo

# Display dataset structure
print("Dataset features:")
print(dataset.features())

# Show first example
print("\nFirst example:")
first_example = dataset[0]
print(f"Article length: {len(first_example['article'])} characters")
print(f"Summary length: {len(first_example['highlights'])} characters")
print("\nArticle snippet:")
print(first_example['article'][:500] + '...')
print("\nSummary:")
print(first_example['highlights'])

## 4. Text Preprocessing

Proper preprocessing is crucial for effective text summarization. Here are the key steps:

### 4.1 Tokenization and Sentence Splitting

Breaking text into sentences and words is the first step. We need to handle punctuation and special cases:

```python
import re

def split_into_sentences(text):
    """Split text into sentences using regex.",
    # Simple sentence splitting based on punctuation
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if s.strip()]
```

### 4.2 Stop Word Removal

Stop words are common words (e.g., 'the', 'is', 'and') that don't carry significant meaning. Removing them can improve summarization quality:

```python
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [word for word in tokens if word.lower() not in stop_words]
```

### 4.3 Punctuation and Special Character Removal

Removing punctuation helps focus on the actual words:

```python
import string

def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))
```

### 4.4 Lemmatization and Stemming

Reducing words to their base or root form improves consistency:

```python
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]
```

### 4.5 Text Cleaning Pipeline

Combining all steps into a comprehensive preprocessing function:

In [ ]:
# Text preprocessing pipeline

import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

def preprocess_text(text, remove_stopwords=True, lemmatize=True):
    """Comprehensive text preprocessing function.
    
    Args:
        text: Input text string
        remove_stopwords: Whether to remove stop words
        lemmatize: Whether to apply lemmatization
    """
    
    # Convert to lowercase
    text = text.lower()

    # Remove special characters and digits
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Tokenize into words
    tokens = word_tokenize(text)

    # Remove stopwords if requested
    if remove_stopwords:
        stop_words = set(stopwords.words('english'))
        tokens = [word for word in tokens if word not in stop_words]

    # Lemmatize if requested
    if lemmatize:
        lemmatizer = WordNetLemmatizer()
        tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return tokens

# Test preprocessing
sample = "Natural language processing is a fascinating field that deals with the interaction between computers and human language."
print("Original:", sample)
print("Preprocessed:", preprocess_text(sample))

## 5. ROUGE Evaluation

ROUGE (Recall-Oriented Understudy for Gisting Evaluation) is the standard metric for evaluating summarization systems. It measures the quality of a summary by comparing it to human-written reference summaries.

### 5.1 ROUGE-N (Overlap of N-grams)

Measures the overlap of n-grams between the generated summary and reference summary. The most common variant is ROUGE-1 (unigrams), ROUGE-2 (bigrams), and ROUGE-L (longest common subsequence).

$$ROUGE-N = \frac{\sum_{S \in \text{References}} \sum_{\text{gram} \in S} \text{Count}(\text{gram})}{\sum_{S \in \text{References}} \sum_{\text{gram} \in S} \text{Count}(\text{gram})\text{ in Reference}}$$

### 5.2 ROUGE-L (Longest Common Subsequence)

Measures the longest matching sequence of words between the summary and reference, considering word order:

$$ROUGE-L = \frac{LCS(C,S)}{LCS(S)}$$\n
### 5.3 ROUGE-Score Components:

- **Precision**: Fraction of summary n-grams that match the reference
- **Recall**: Fraction of reference n-grams that match the summary
- **F1-score**: Harmonic mean of precision and recall

### 5.4 Using ROUGE in Python

In [ ]:
# ROUGE evaluation

from rouge import Rouge

# Example summaries
hypothesis = "TextRank is an extractive summarization method based on graph algorithms."
reference = "TextRank is a graph-based ranking algorithm used for extractive text summarization."

# Calculate ROUGE scores
rouge = Rouge()
scores = rouge.get_scores(hypothesis, reference)

print("ROUGE Scores:")
print(f"ROUGE-1 F1: {scores[0]['rouge-1']['f']:.4f}")
print(f"ROUGE-2 F1: {scores[0]['rouge-2']['f']:.4f}")
print(f"ROUGE-L F1: {scores[0]['rouge-l']['f']:.4f}")

# Detailed breakdown
print("\nDetailed scores:")
print(f"ROUGE-1 Precision: {scores[0]['rouge-1']['p']:.4f}")
print(f"ROUGE-1 Recall: {scores[0]['rouge-1']['r']:.4f}")
print(f"ROUGE-2 Precision: {scores[0]['rouge-2']['p']:.4f}")
print(f"ROUGE-2 Recall: {scores[0]['rouge-2']['r']:.4f}")
print(f"ROUGE-L Precision: {scores[0]['rouge-l']['p']:.4f}")
print(f"ROUGE-L Recall: {scores[0]['rouge-l']['r']:.4f}")

## 6. Comparison with Frequency-Based Methods

Before TextRank, many extractive summarization systems were based on word frequency analysis. Let's compare TextRank with a simple frequency-based approach.

### 6.1 Frequency-Based Summarization

The simplest approach is to select sentences containing the most frequent words (after removing stop words). Here's an implementation:

In [ ]:
# Frequency-based summarization

def frequency_summarize(text, num_summaries=3):
    """Extractive summarization based on word frequency.
    
    Args:
        text: Input text
        num_summaries: Number of sentences to include
    """
    
    # Split into sentences
    sentences = [s.strip() for s in text.split('. ') if s]

    # Preprocess and count word frequencies
    word_freq = Counter()
    for sentence in sentences:
        words = preprocess_text(sentence, remove_stopwords=True, lemmatize=False)
        word_freq.update(words)

    # Score sentences based on word frequencies
    sentence_scores = []
    for i, sentence in enumerate(sentences):
        words = preprocess_text(sentence, remove_stopwords=True, lemmatize=False)
        if words:
            score = sum(word_freq[word] for word in words) / len(words)
            sentence_scores.append((score, i, sentence))

    # Select top-k sentences
    sentence_scores.sort(reverse=True)
    top_indices = [idx for _, idx, _ in sentence_scores[:num_summaries]]
    top_indices.sort()
    
    summary = '. '.join([sentences[i] for i in top_indices]) + '.'
    return summary, sentence_scores[:num_summaries]

# Compare both methods
print("Comparing TextRank vs Frequency-based summarization:\n")

sample_text = """.
Machine learning is a subset of artificial intelligence that enables computers to learn from data without being explicitly programmed.
Deep learning, a branch of machine learning, uses neural networks with many layers to model complex patterns.
Natural language processing combines computational linguistics with statistical methods to process human language.
Text summarization helps manage the overwhelming amount of textual information available today.
Extractive methods like TextRank select important sentences directly from the document.
Frequency-based methods simply pick sentences with the most common words.
Graph-based methods consider the relationships between sentences.
The CNN/Daily Mail dataset is widely used for training summarization models.
ROUGE scores are used to evaluate summary quality.
"""

print("Original text has {} sentences".format(len([s for s in sample_text.split('. ') if s.strip()])))

# TextRank summary
tr_summary, tr_scores = textrank_summarize(sample_text, num_summaries=3)
print("\nTextRank Summary:")
print(tr_summary)

# Frequency-based summary
freq_summary, freq_scores = frequency_summarize(sample_text, num_summaries=3)
print("\nFrequency-based Summary:")
print(freq_summary)

## 7. Key Differences and When to Use Each Method

| Aspect | TextRank | Frequency-Based | 
|--------|----------|----------------|
| **Approach** | Graph-based, considers sentence relationships | Statistical, based on word occurrence | 
| **Context** | Considers global structure of the document | Local word frequency only | 
| **Quality** | Generally higher quality, more coherent | May pick redundant or less important info | 
| **Computational Cost** | Higher (similarity matrix, graph algorithms) | Lower (simple counting) | 
| **Flexibility** | Can incorporate different similarity measures | Very simple to implement and understand | 

### When to Use Each:

- **Frequency-based**: Quick and dirty summarization, educational purposes, when computational resources are extremely limited
- **TextRank**: Production systems, better quality summaries, when you need coherent and non-redundant summaries

### Limitations of Both:

Both are **extractive** methods and have inherent limitations:
- They can only select sentences that exist in the original text
- They may produce summaries that are grammatically correct but not fluent
- They struggle with documents containing multiple topics or themes

Modern abstractive methods using sequence-to-sequence models (like T5, BART, PEGASUS) trained on CNN/Daily Mail have largely superseded these traditional approaches for state-of-the-art results.

## 8. Conclusion

In this notebook, we've explored:

1. **TextRank algorithm**: A graph-based extractive summarization method
2. **CNN/Daily Mail dataset**: The standard benchmark for summarization research
3. **Text preprocessing**: Essential steps including tokenization, stop word removal, and lemmatization
4. **ROUGE evaluation**: The standard metric for automatic summarization evaluation
5. **Comparison with frequency-based methods**: TextRank generally produces higher quality summaries by considering sentence relationships

While these traditional methods are educational and useful for understanding the fundamentals, modern abstractive approaches using large language models have become the state-of-the-art for text summarization tasks.